# Feature Engineering

This notebook transforms the prepared bike-sharing dataset into the final feature matrix used for demand prediction.

The main goals are:

- Create time-based features.
- Create weekend and rush-hour indicators.
- Remove columns that should not be used as predictors.
- Separate the feature matrix (`X`) from the target (`y`).
- Verify the final feature set and prevent target leakage.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.data_processing import load_and_prepare_data
from src.feature_engineering import (
    BASE_FEATURES,
    ENGINEERED_FEATURES,
    create_features,
)

## 1. Load the Prepared Dataset

In [2]:
DATA_PATH = Path("../data/raw/hour.csv")

df = load_and_prepare_data(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (17379, 17)


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


## 2. Original Columns

The raw dataset contains identifiers, temporal variables, weather variables, user-type counts, and the total rental count.

The following columns require special treatment:

- `instant` is an identifier.
- `casual` and `registered` are components of the target `cnt` and must not be used as predictors.

In [3]:
print("Original columns:")
print(df.columns.tolist())

Original columns:
['instant', 'dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'casual', 'registered', 'cnt']


## 3. Create Time-Based Features

The feature engineering module extracts:

- `year`
- `month`
- `day`
- `hour`
- `weekday`

It also creates:

- `is_weekend`
- `is_rush_hour`

In [4]:
from src.feature_engineering import add_time_features

featured_df = add_time_features(df)

featured_df[
    [
        "dteday",
        "hr",
        "year",
        "month",
        "day",
        "hour",
        "weekday",
        "is_weekend",
        "is_rush_hour",
    ]
].head(10)

,dteday,hr,year,month,day,hour,weekday,is_weekend,is_rush_hour
0,2011-01-01,0,2011,1,1,0,6,1,0
1,2011-01-01,1,2011,1,1,1,6,1,0
2,2011-01-01,2,2011,1,1,2,6,1,0
3,2011-01-01,3,2011,1,1,3,6,1,0
4,2011-01-01,4,2011,1,1,4,6,1,0
5,2011-01-01,5,2011,1,1,5,6,1,0
6,2011-01-01,6,2011,1,1,6,6,1,0
7,2011-01-01,7,2011,1,1,7,6,1,1
8,2011-01-01,8,2011,1,1,8,6,1,1
9,2011-01-01,9,2011,1,1,9,6,1,1


### Weekend Indicator

`is_weekend` is set to `1` when the day is Saturday or Sunday and `0` otherwise.

The original `weekday` feature is retained as well because it contains more detailed weekly information.

In [5]:
featured_df[
    ["weekday", "is_weekend"]
].drop_duplicates().sort_values("weekday")

,weekday,is_weekend
24,0,1
47,1,0
69,2,0
92,3,0
115,4,0
138,5,0
0,6,1


### Rush-Hour Indicator

For this project, rush hour is defined as:

- 07:00
- 08:00
- 09:00
- 16:00
- 17:00
- 18:00
- 19:00

This feature is intended to capture periods of potentially higher commuting demand.

In [6]:
featured_df[
    ["hour", "is_rush_hour"]
].drop_duplicates().sort_values("hour")

,hour,is_rush_hour
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0
5,5,0
6,6,0
7,7,1
8,8,1
9,9,1


## 4. Create the Final Feature Matrix

The final predictors contain:

### Base features

- season
- holiday
- workingday
- weathersit
- temp
- hum
- windspeed

### Engineered features

- year
- month
- day
- hour
- weekday
- is_weekend
- is_rush_hour

The target is `cnt`.

In [7]:
X, y = create_features(df)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (17379, 14)
y shape: (17379,)


In [8]:
print("Final features:")
print(X.columns.tolist())

Final features:
['season', 'holiday', 'workingday', 'weathersit', 'temp', 'hum', 'windspeed', 'year', 'month', 'day', 'hour', 'weekday', 'is_weekend', 'is_rush_hour']


## 5. Verify Target Leakage

The following columns must not appear in the feature matrix:

- `instant`
- `casual`
- `registered`
- `cnt`

`cnt` is the prediction target, while `casual` and `registered` directly compose `cnt`.

Therefore, including them as predictors would introduce target leakage.

In [9]:
excluded_columns = [
    "instant",
    "casual",
    "registered",
    "cnt",
]

leakage_columns = [
    column
    for column in excluded_columns
    if column in X.columns
]

print("Excluded columns found in X:", leakage_columns)

assert not leakage_columns

Excluded columns found in X: []


## 6. Feature Groups

The final feature matrix contains 14 predictors.

The model preprocessing stage will treat categorical and numerical variables differently.

In [10]:
print("Base features:")
print(BASE_FEATURES)

print("\nEngineered features:")
print(ENGINEERED_FEATURES)

print("\nTotal features:", len(BASE_FEATURES) + len(ENGINEERED_FEATURES))

Base features:
['season', 'holiday', 'workingday', 'weathersit', 'temp', 'hum', 'windspeed']

Engineered features:
['year', 'month', 'day', 'hour', 'weekday', 'is_weekend', 'is_rush_hour']

Total features: 14


In [11]:
X.head()

,season,holiday,workingday,weathersit,temp,hum,windspeed,year,month,day,hour,weekday,is_weekend,is_rush_hour
0,1,0,0,1,0.24,0.81,0.0,2011,1,1,0,6,1,0
1,1,0,0,1,0.22,0.80,0.0,2011,1,1,1,6,1,0
2,1,0,0,1,0.22,0.80,0.0,2011,1,1,2,6,1,0
3,1,0,0,1,0.24,0.75,0.0,2011,1,1,3,6,1,0
4,1,0,0,1,0.24,0.75,0.0,2011,1,1,4,6,1,0


## Conclusion

Feature engineering is complete.

The final dataset contains:

- 17,379 observations.
- 14 predictor variables.
- `cnt` as the target variable.

The feature matrix excludes the record identifier and the variables that directly compose the target.

The resulting features will be used in the demand prediction stage.